In [ ]:
# | default_exp index_tracking


# Index Tracking
> Inspect, store, and analyze Google Search Console URL indexing status.

In [ ]:
# | export
from sqlmodel import Session, select
from seootter.models import IndexStatus
from seootter.gsc_client import GSCAuth
from googleapiclient.discovery import build
from datetime import datetime
import httpx
from bs4 import BeautifulSoup
import time

from seootter.sqlite_db import get_session

In [ ]:
# | export
def inspect_url_status(auth: GSCAuth,  # Authenticated GSCAuth instance
                       site_url: str,  # GSC property URL
                       page_url: str  # Page URL to inspect
                       ) -> dict:
    "Inspect URL indexing status from Google Search Console."
    service = build("searchconsole", "v1", credentials=auth.get_credentials())
    response = service.urlInspection().index().inspect(
        body={"inspectionUrl": page_url, "siteUrl": site_url, "languageCode": "en-US"}
    ).execute()
    result = response.get("inspectionResult", {}).get("indexStatusResult", {})
    return {"verdict": result.get("verdict", "UNKNOWN"),
            "coverage_state": result.get("coverageState"),
            "last_crawl_time": result.get("lastCrawlTime"),
            "indexing_state": result.get("indexingState"),
            "robots_txt_state": result.get("robotsTxtState")}


In [ ]:
# | test
#| eval: false
from fastcore.test import test_eq

from pprint import pprint
from seootter.gsc_client import GSCAuth
from sqlmodel import Session, create_engine, SQLModel
from seootter.sqlite_db import get_session

auth = GSCAuth()

# Test inspect_url_status
status = inspect_url_status(auth, "sc-domain:kareemai.com", "https://kareemai.com/")
print(f"Verdict: {status['verdict']}")
test_eq("verdict" in status, True)
pprint(status)

In [ ]:
# | export
def store_index_status(session: Session,  # Active database session
                       auth: GSCAuth,  # Authenticated GSCAuth instance
                       site_url: str,  # GSC property URL
                       page_url: str  # Page URL to inspect and store
                       ) -> None:
    "Inspect and store URL index status as a new history row."
    record = IndexStatus(site_url=site_url, page_url=page_url,
                         **inspect_url_status(auth, site_url, page_url))
    session.add(record)
    session.commit()


In [ ]:
# | hide
#| eval: false

with get_session() as session:
    store_index_status(session, auth, "sc-domain:kareemai.com", "https://kareemai.com/")
    print("Stored in ./data/seo.db")

    pprint(status)


In [ ]:
# | export
def get_index_status(session: Session,  # Active database session
                     site_url: str,  # GSC property URL
                     verdict: str | None = None  # Optional verdict to filter by
                     ) -> list[IndexStatus]:
    "Get latest index status per page, optionally filtered by verdict."
    from sqlalchemy import func
    latest = (select(IndexStatus.page_url, func.max(IndexStatus.checked_at).label("max_checked"))
              .where(IndexStatus.site_url == site_url)
              .group_by(IndexStatus.page_url).subquery())
    query = select(IndexStatus).join(
        latest, (IndexStatus.page_url == latest.c.page_url) &
                (IndexStatus.checked_at == latest.c.max_checked))
    if verdict: query = query.where(IndexStatus.verdict == verdict)
    return session.exec(query).all()


In [ ]:
# | hide
#| eval: false

test_index_status = get_index_status(
    session,
    "sc-domain:kareemai.com",
)
pprint(test_index_status)

In [ ]:
# | export
def get_not_indexed_pages(session: Session,  # Active database session
                          site_url: str  # GSC property URL
                          ) -> list[IndexStatus]:
    "Get pages whose latest index status is not PASS."
    return [p for p in get_index_status(session, site_url) if p.verdict != "PASS"]


In [ ]:
#| export
def get_not_indexed_by_reason(session: Session,  # Active database session
                              site_url: str  # GSC property URL
                              ) -> dict[str, list[IndexStatus]]:
    "Group not-indexed pages by their coverage state reason."
    from collections import defaultdict
    grouped = defaultdict(list)
    for page in get_not_indexed_pages(session, site_url):
        grouped[page.coverage_state or "Unknown"].append(page)
    return dict(grouped)

In [ ]:
# | export
def fetch_sitemap_urls(sitemap_url: str  # URL of the sitemap XML
                       ) -> list[str]:
    "Fetch all page URLs from a sitemap XML. Returns empty list on connection error."
    try:
        soup = BeautifulSoup(httpx.get(sitemap_url, timeout=30).text, "xml")
        return [loc.text for loc in soup.find_all("loc")]
    except (httpx.ConnectError, httpx.TimeoutException, httpx.HTTPStatusError):
        return []


In [ ]:
# | export
def store_all_index_status(session: Session,  # Active database session
                           auth: GSCAuth,  # Authenticated GSCAuth instance
                           site_url: str,  # GSC property URL
                           sitemap_url: str  # URL of the sitemap to process
                           ) -> dict:
    "Check and store index status for all pages in a sitemap."
    urls = fetch_sitemap_urls(sitemap_url)
    results = {"successful": [], "failed": []}
    for i, url in enumerate(urls, 1):
        print(f"Checking {i}/{len(urls)}: {url}")
        try:
            store_index_status(session, auth, site_url, url)
            results["successful"].append(url)
        except Exception as e:
            results["failed"].append({"url": url, "error": str(e)})
        time.sleep(1)
    return results

In [ ]:
#| hide
#| eval: false
reasons = get_not_indexed_by_reason(session, "sc-domain:kareemai.com")
for reason, pages in reasons.items():
    print(f"\n{reason} ({len(pages)} pages):")
    for p in pages:
        print(f"  {p.page_url}")
reasons

In [ ]:
# | hide
#| eval: false
test_storing_all_index_status = store_all_index_status(
    session,
    auth,
    site_url="sc-domain:kareemai.com",
    sitemap_url="http://127.0.0.1:6287/sitemap.xml",
)
test_storing_all_index_status

In [ ]:
# | hide
#| eval: false
with get_session() as session:
    test_getting_not_indexed = get_not_indexed_pages(
        session, site_url="sc-domain:kareemai.com"
    )
    pprint(test_getting_not_indexed)

In [ ]:
#| export
def get_index_history(session: Session,  # Active database session
                      page_url: str  # Page URL to look up
                      ) -> list[IndexStatus]:
    "Get full index status history for a page, newest first."
    return session.exec(
        select(IndexStatus).where(IndexStatus.page_url == page_url)
        .order_by(IndexStatus.checked_at.desc())
    ).all()

In [ ]:
#|hide
#| eval: false
history = get_index_history(session, "https://www.smaagarden.com")
for h in history:
    print(f"{h.checked_at} | {h.verdict} | {h.coverage_state}")
